In [ ]:
import os
import re
import json
import ast
import string
import unicodedata
from collections import Counter

import pandas as pd
import torch
import difflib
import pkg_resources
from tqdm import tqdm

import pyterrier as pt
from pyterrier.measures import R, MAP

from keybert import KeyBERT
from symspellpy import SymSpell, Verbosity
from transformers import pipeline
from pathlib import Path

# Initialization

In [ ]:
UPLOAD_VARIABLES= True
BASE_DIR = Path.cwd()

## Download the Collection

In [302]:
if not UPLOAD_VARIABLES:
    !curl -L "https://huggingface.co/datasets/Bhawna/ChroniclingAmericaQA/resolve/main/test.json?download=true" -o test.json
    !curl -L "https://huggingface.co/datasets/Bhawna/ChroniclingAmericaQA/resolve/main/train.json?download=true" -o train.json
    !curl -L "https://huggingface.co/datasets/Bhawna/ChroniclingAmericaQA/resolve/main/dev.json?download=true" -o validation.json

    
    files = ["train.json", "validation.json", "test.json"]

    for path in files:
        print(f"\n===== {path} =====")
        try:
            with open(path, "r", encoding="utf-8") as f:
                # Read a few hundred characters to see what kind of JSON it is
                head = f.read(500)
                print("Preview of first 500 characters:\n")
                print(head[:500])
            # Try to load only part of the file
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list):
                print(f"\nLoaded {len(data)} items (list).")
                print("Dictionary keys:", list(data[0].keys()))
                print(json.dumps(data[0], indent=2)[:600])
            elif isinstance(data, dict):
                print("\nTop-level is a dictionary. Keys:", list(data.keys()))
                for k, v in data.items():
                    if isinstance(v, list):
                        print(f"Key '{k}' contains a list of {len(v)} items.")
                        if v:
                            print("First item keys:", list(v[0].keys()))
                            print(json.dumps(v[0], indent=2)[:600])
                            break
            else:
                print(f"Unexpected top-level type: {type(data)}")
        except Exception as e:
            print(f"Could not parse {path} as JSON: {e}")

## Create the Document Collection

In [303]:
if not UPLOAD_VARIABLES:

    inputs = ["train.json", "validation.json", "test.json"]
    output = "document_collection.json"

    def load_list_or_empty(path):
        if not os.path.exists(path) or os.path.getsize(path) == 0:
            print(f"Skipping {path} because it is missing or empty")
            return []
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list):
                return data
            print(f"Skipping {path} because it is not a list at the top level")
            return []
        except json.JSONDecodeError:
            print(f"Skipping {path} because it is not valid JSON")
            return []

    def project(recs):
        out = []
        for r in recs:
            out.append({
                "para_id": r.get("para_id", ""),
                "context": r.get("context", ""),
                "raw_ocr": r.get("raw_ocr", ""),
                "publication_date": r.get("publication_date", "")
            })
        return out

    all_recs = []
    for p in inputs:
        recs = load_list_or_empty(p)
        print(f"Loaded {len(recs)} records from {p}")
        all_recs.extend(project(recs))

    # deduplicate by para_id keeping the first one seen
    uniq = {}
    for rec in all_recs:
        pid = rec.get("para_id", "")
        if pid and pid not in uniq:
            uniq[pid] = rec

    result = list(uniq.values())

    with open(output, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"Wrote {len(result)} records to {output}")
    print(json.dumps(result[:3], indent=2))

## Create the Test Queries

In [304]:
if not UPLOAD_VARIABLES:

    input_file = "test.json"
    output_file = "test_queries.json"

    # Load the data
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    def clean_question(text):
        if not isinstance(text, str):
            return ""
        text = unicodedata.normalize("NFKC", text)
        text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)  # remove punctuation
        text = re.sub(r"\s+", " ", text)  # collapse multiple spaces
        return text.strip()

    # Extract and clean
    queries = [
        {
            "query_id": item.get("query_id", ""),
            "question": clean_question(item.get("question", "")),
        }
        for item in data
    ]

    # Sort by query_id (assuming numeric)
    queries = sorted(queries, key=lambda x: int(x["query_id"]) if str(x["query_id"]).isdigit() else x["query_id"])

    # Keep only the first 10,000
    queries = queries[:10000]

    # Save new JSON
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(queries, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(queries)} entries to {output_file}")
    print(json.dumps(queries[:3], indent=2))

## Create the QRels

In [305]:
if not UPLOAD_VARIABLES:
    input_file = "test.json"
    qrels_file = "test_qrels.json"
    answers_file = "test_query_answers.json"

    # Load the data
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Build the qrels file: query_id, iteration=0, para_id, relevance=1
    qrels = [
        {
            "query_id": item.get("query_id", ""),
            "iteration": 0,
            "para_id": item.get("para_id", ""),
            "relevance": 1
        }
        for item in data
    ]

    # Build the query_answers file: same plus answer and org_answer
    query_answers = [
        {
            "query_id": item.get("query_id", ""),
            "iteration": 0,
            "para_id": item.get("para_id", ""),
            "relevance": 1,
            "answer": item.get("answer", ""),
            "org_answer": item.get("org_answer", "")
        }
        for item in data
    ]

    # Save both files
    with open(qrels_file, "w", encoding="utf-8") as f:
        json.dump(qrels, f, ensure_ascii=False, indent=2)

    with open(answers_file, "w", encoding="utf-8") as f:
        json.dump(query_answers, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(qrels)} entries to {qrels_file}")
    print(f"Saved {len(query_answers)} entries to {answers_file}")
    print("Sample qrels entry:", qrels[0])
    print("Sample query_answers entry:", query_answers[0])

In [306]:
if not UPLOAD_VARIABLES:
    #  Convert the list of dictionaries into a pandas DataFrame
    queries_df = pd.DataFrame(queries)
    qrels_df = pd.DataFrame(qrels)

    # Filter out rows where the 'question' is now empty
    queries_df = queries_df[queries_df['question'].str.len() > 0]

    # Display the cleaned DataFrame
    print("Queries DataFrame:")
    print(queries_df)
    print("Qrels DataFrame:")
    print(qrels_df)


    # 1. Load the JSON file
    input_file = "document_collection.json"

    if os.path.exists(input_file):
        with open(input_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        # 2. Convert to DataFrame
        df = pd.DataFrame(data)

        # 3. Select ONLY 'para_id' and 'context'
        df = df[["para_id", "context"]]

        # 4. RENAME columns to match what PyTerrier needs
        df = df.rename(columns={
            "para_id": "docno"
        })

        # 5. Type Safety (Make sure they are strings)
        df["docno"] = df["docno"].astype(str)
        df["context"] = df["context"].astype(str)

        print(f"✅ Dataset created with {len(df)} documents.")
        print(df.head())


        # Force IDs to be strings in both dataframes , this is very important for pyterrier to read things correctly
    queries_df = queries_df.assign(
        qid = queries_df['query_id'].astype(str),
        query = queries_df['question'].astype(str)
    )

    qrels_df = qrels_df.assign(
        qid = qrels_df['query_id'].astype(str),
        docno = qrels_df['para_id'].astype(str),
        label = qrels_df['relevance'].astype(int)
)

# OFFLINE Initialization
Suggested initialization that uses files provided with the script. It should grand a smooth and error-free execution

In [ ]:
if UPLOAD_VARIABLES:
    
    queries_df= pd.read_csv(str(BASE_DIR / "files" / "queries.csv"))
    qrels_df= pd.read_csv(str(BASE_DIR / "files" / "qrels.csv"))


    index_path_context = BASE_DIR / "indices" / "index_context"
    index_path_fuzzy   = BASE_DIR / "indices" / "index_fuzzy"
    index_path_cleaned = BASE_DIR / "indices" / "index_cleaned"
    index_path_multi   = BASE_DIR / "indices" / "index_multi"

    index_fuzzy= pt.IndexFactory.of(str(index_path_fuzzy))
    index_baseline= pt.IndexFactory.of(str(index_path_context))
    index_correction= pt.IndexFactory.of(str(index_path_cleaned))
    index_multi= pt.IndexFactory.of(str(index_path_multi))
    df= pd.read_csv(str(BASE_DIR / "files" / "entity_doc.csv"))

20:54:52.168 [main] WARN org.terrier.structures.FSADocumentIndex -- This index has fields, but FSADocumentIndex is used (which stores fields lengths on disk); If using field-based models such as BM25F, change to index.document.class in the index  properties file to FSAFieldDocumentIndex or FSADocumentIndexInMemFields to support efficient retrieval. If you don't use (e.g.) BM25F, this warning can be ignored


# Retrieval

In [308]:
import pyterrier as pt
import pandas as pd
import re 

import os
##If we are running on our personal device, we need to initialize java.
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-25'
os.environ['_JAVA_OPTIONS'] = '-Xmx14g'
print("JVM Memory Limit set via _JAVA_OPTIONS: 14GB")
import pyterrier as pt
if not pt.started():
    pt.init() 

JVM Memory Limit set via _JAVA_OPTIONS: 14GB


C:\Users\samue\AppData\Local\Temp\ipykernel_3948\2169879283.py:11: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


The following functions creates character n-grams after removing stopwords, it takes a string as input and return the n-gram tokenized string as output.

## Helper Functions

In [ ]:
STOPWORDS = set(["the", "and", "is", "of", "in", "to", "a", "it", "that", "for", "on", 
                 "with", "as", "by", "at", "an", "be", "this", "which", "or", "from", 
                 "what", "where", "when", "who", "how", "why"])

def to_ngrams(text, n=3):
    """
    Converts a string into a space-separated sequence of character n-grams
    """
    if pd.isna(text) or text == "":
        return ""
    
    text = str(text).lower()

    # Remove non-alphanumeric chars
    text = re.sub(r'[^a-z0-9 ]', ' ', text)

    # Split into words
    words = text.split()
    
    # Check against the STOPWORDS set
    # clean_words = [w for w in words if w not in STOPWORDS and len(w) > 1]
    clean_words = [w for w in words if len(w) > 1]

    # Join back together
    clean_text = " ".join(clean_words)

    # add padding to shorter words
    if len(clean_text) < n:
        return clean_text + ("_" * (n - len(clean_text)))
  
    # Generate n-grams
    ngrams = [clean_text[i : i+n] for i in range(len(clean_text) - n + 1)]

    # Replace spaces with underscores
    valid_ngrams = [x.replace(" ", "_") for x in ngrams]

    return " ".join(valid_ngrams)

def format_5g(row): return to_ngrams(row['question'], n=5)

the following function apply the n-gram transformation to a query

The following function is used to avoid code repetitions, it is used to create indices in one liners

In [310]:
def to_index(path,meta,attributes,stemming=True,df=df):
    text_fields=[attributes]
    temp=df
    docs= temp[[meta,attributes]].to_dict('records')
    metas= {meta : 50}
    indexer = pt.index.IterDictIndexer(path,meta=metas, text_attrs=text_fields, overwrite=True)
    if stemming==False:
        indexer.setProperty("termpipelines", "") 
        indexer.setProperty("tokeniser", "WhitespaceTokeniser")
        
    indexref = indexer.index(docs)
    index = pt.IndexFactory.of(indexref) #our index
    return index

### DYNAMIC QUERY REWRITING

The objective of this module is to adjust query weights (or in our specific case term frequencies).
The idea is that different type of queries have different information cues, which are carried by only a portion ot it.
Insthead of manually adjusting the weights based on the query type (who, what, where) and some euristhic, we decided to employ a smarter KeyWord extraction method throught KeyBert

In [ ]:
kw_model = KeyBERT()

def get_keybert_weights(query):
    # extract keywords
    keywords = kw_model.extract_keywords(
        query, 
        keyphrase_ngram_range=(1, 1), 
        stop_words='english',
        top_n=5
    )
    
    weighted_parts = []
    seen_words = set()
    
    for word, score in keywords:
        # we scale the score between 1 and 10
        raw_weight = score * 10
        
        # round to closest integer
        weight_int = int(round(raw_weight))
        
        # clamp to ensure it stays strictly 1-10 (just in case)
        weight_int = max(1, min(20, weight_int))
        
        weighted_parts.append(f"{word}^{weight_int}")
        seen_words.add(word)
        
    #word not extracted as keyword get assigned weight 1
    for word in query.split():
        clean_word = word.lower() 
        if clean_word not in seen_words:
            weighted_parts.append(f"{clean_word}^1.0")
            
    return " ".join(weighted_parts)




#NORMAL QUERY WEIGHTING SYNTAX GIVES PROBLEMS AND WORSEN PERFORMANCES, WE FOUND OUT JUST REPEATING TERMS YEAL BETTER RESULTS (we adjust bm25 query saturation too)
def expand_weights_to_repetitions(query):
    if not isinstance(query, str): return ""
    
    #I do not want alphanumeric characters
    pattern = r'("[^"]*"|[^"\s^]+)\^(\d+)'
    
    # we use a callback function to replace the match with the repeated string
    def replace_func(match):
        term = match.group(1)   
        count = int(match.group(2)) 
        
        # Repeat the term 'count' times, joined by space
        return " ".join([term] * count)
    
    # 2. Apply the replacement
    # re.sub can take a function as the second argument!
    expanded_query = re.sub(pattern, replace_func, query)
    
    return expanded_query



    

Original: Who was the capitaine of the corvette Savannah
Weighted: corvette^10 savannah^9 capitaine^7 who^1.0 was^1.0 the^1.0 of^1.0 the^1.0


### ENTITY COLLAPSING

Entity are retrieved as touples (Entity,Type), for our task we just need the entity

In [ ]:
def extract_entity_names(entities):
    """
    Input:
      entities: list of tuples like [(name, label), ...]
    Output:
      list of entity names only
    """
    if not isinstance(entities, list):
        return []

    names = []
    for item in entities:
        # tuple/list case: ("France", "GPE") or ("France","GPE",...)
        if isinstance(item, (list, tuple)) and len(item) > 0:
            if isinstance(item[0], str) and item[0].strip():
                names.append(item[0])

        # fallback: plain string (rare but safe)
        elif isinstance(item, str) and item.strip():
            names.append(item)

    return names


def extract_kw(row): return get_keybert_weights(row['query'])
def transform_w(row): return expand_weights_to_repetitions(row['query']) #we need to collapse the weights into repetition, otherwise it won't work well

### QUERY CLEANING

To clean and modernize the spelling, we first remove all non alphanumeric character, which origin is mostly due to OCR mistakes, and in general are not important retrieval cue. Secondly, we start analyzing our contexts , targetting words with low zipf frequency. If those words are entities (precedently extracted) or have an high frequency inbetween our collection, they are ignored. Otherwise a candidate substitution as an edit distance of one is looked for in our modern corpus. if more than one alternative is present, a Bert model is employed to select the most probable candidate given the context throught Bert fill-masks.

In [343]:

from wordfreq import zipf_frequency

class SmartHistoricalCleaner:
    def __init__(
        self,
        # all_documents_text is no longer strictly needed for frequency map, 
        # but kept here for signature compatibility.
        all_documents_text=None, 
        zipf_threshold: float = 1.0, # changed from trust_threshold
        use_bert: bool = True,
        max_bert_chars: int = 512,
        max_edit_distance: int = 1,
        min_token_len: int = 4,
    ):
        self.zipf_threshold = float(zipf_threshold)
        self.use_bert = bool(use_bert)
        self.max_bert_chars = int(max_bert_chars)
        self.max_edit_distance = int(max_edit_distance)
        self.min_token_len = int(min_token_len)

        # 1. Load SymSpell
        print("Loading SymSpell...")
        self.sym_spell = SymSpell(max_dictionary_edit_distance=self.max_edit_distance, prefix_length=7)
        dictionary_path = pkg_resources.resource_filename(
            "symspellpy", "frequency_dictionary_en_82_765.txt"
        )
        ok = self.sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)
        if not ok:
            raise RuntimeError("Failed to load SymSpell dictionary.")

        # 2. Load BERT
        if self.use_bert:
            print("Loading BERT fill-mask...")
            device = 0 if torch.cuda.is_available() else -1
            self.bert = pipeline("fill-mask", model="distilbert-base-uncased", device=device)
            self._mask = self.bert.tokenizer.mask_token
        else:
            self.bert = None
            self._mask = None
            
        # Note: Corpus frequency map building removed as we now use global Zipf scores.

    # -------------------------
    # Utilities
    # -------------------------
    @staticmethod
    def sanitize_text(text: str) -> str:
        if not isinstance(text, str):
            return ""
        text = re.sub(r"[^A-Za-z0-9 \n]+", " ", text)
        lines = text.split("\n")
        lines = [re.sub(r"\s+", " ", ln).strip() for ln in lines]
        return "\n".join(lines)

    @staticmethod
    def _normalize_entities_list(entities_list):
        if not entities_list:
            return []
        out = []
        for e in entities_list:
            if isinstance(e, str):
                out.append(e)
            elif isinstance(e, (list, tuple)) and len(e) > 0 and isinstance(e[0], str):
                out.append(e[0])
        return out

    def _build_protected_token_set(self, entities_list) -> set:
        ents = self._normalize_entities_list(entities_list)
        protected = set()
        for ent in ents:
            ent = self.sanitize_text(ent)
            if not ent:
                continue
            for tok in ent.split():
                if tok:
                    protected.add(tok.lower())
        return protected

    def _bert_choose(self, line: str, target: str, candidates):
        if not self.use_bert or not self.bert:
            return None
        if not line or not target or not candidates:
            return None

        escaped = re.escape(target)
        pat = rf"\b{escaped}\b"
        if re.search(pat, line):
            masked = re.sub(pat, self._mask, line, count=1)
        else:
            masked = line.replace(target, self._mask, 1)

        masked = masked[: self.max_bert_chars]

        try:
            preds = self.bert(masked, top_k=10)
        except Exception:
            return None

        bert_tokens = [p["token_str"].strip().lower() for p in preds if "token_str" in p]
        cand_l = [c.lower() for c in candidates]
        for bt in bert_tokens:
            if bt in cand_l:
                return candidates[cand_l.index(bt)]
        return None

    # -------------------------
    # Main Logic
    # -------------------------
    def clean_text(self, text, entities_list=None) -> str:
        if not isinstance(text, str):
            return ""

        text = self.sanitize_text(text)
        protected_tokens = self._build_protected_token_set(entities_list)
        
        out_lines = []
        for line in text.split("\n"):
            if not line.strip():
                out_lines.append(line)
                continue

            words = line.split()
            cleaned = []

            for w in words:
                # Remove non-alphanumeric
                w = re.sub(r"[^A-Za-z0-9]+", "", w)
                if not w:
                    continue

                wl = w.lower()

                # 1. Protection Check
                if wl in protected_tokens:
                    cleaned.append(w)
                    continue

                # 2. Global Frequency Check (Zipf)
                z_score = zipf_frequency(wl, 'en')
                
                # Trust filters:
                # If Zipf >= Threshold (common), or too short, or has digits -> Keep it
                if (
                    z_score >= self.zipf_threshold
                    or len(w) < self.min_token_len
                    or not w.isalpha()
                ):
                    cleaned.append(w)
                    continue

                # 3. SymSpell Lookup
                suggestions = self.sym_spell.lookup(
                    wl,
                    Verbosity.TOP,
                    max_edit_distance=self.max_edit_distance,
                    include_unknown=False,
                )

                if not suggestions:
                    cleaned.append(w)
                    continue

                cands = [s.term for s in suggestions]

                # --- NEW LOGIC START ---
                # If Zipf > 0, the word exists in the language (but is rare/below threshold).
                # We add it to candidates so BERT can choose to keep it.
                # If Zipf == 0, it is likely garbage/typo, so we exclude it to force a correction.
                if z_score > 0:
                    if wl not in cands:
                        cands.append(wl)
                # --- NEW LOGIC END ---

                if len(cands) == 1:
                    cleaned.append(cands[0])
                else:
                    # Let BERT choose the best fit among candidates (including original if applicable)
                    best = self._bert_choose(line, w, cands)
                    cleaned.append(best if best else w)

            out_lines.append(" ".join(cleaned))

        return "\n".join(out_lines)




## PREPROCESSING

We create a copy of the 'context' field in 5-grams

In [314]:
if not UPLOAD_VARIABLES:
    df['fuzzy_context'] = df['context'].fillna("").astype(str).apply(lambda x: to_ngrams(x, n=5))

### ENTITY EXTRACTION

we use spacy transformer model to extract entities and their type

In [ ]:
if not UPLOAD_VARIABLES:
    spacy.require_gpu()
    nlp = spacy.load("en_core_web_trf") #download this model

    # 3. Process
    print(f"Processing {len(df)} documents on GPU...")

    results = []
    # Batch size 50-100 is usually safe for T4 GPU with this model
    for doc in tqdm(nlp.pipe(df['context'].astype(str), batch_size=50), total=len(df)):
        # Extract entities
        ents = [(e.text, e.label_) for e in doc.ents]
        results.append(ents)

    # 4. Save
    df['entities'] = results
    df["entity_names"] = df["entities"].apply(extract_entity_names) 
    

### CONTEXT CORRECTION
We apply the correction to the context as described previously

In [344]:
if UPLOAD_VARIABLES:

    # --- 0) Basic safety checks ---
    required_cols = {"context", "entities"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"documents_ent is missing required columns: {missing}")

    # --- 1) Ensure entities are actual Python lists (not strings) ---
    # Handle NaN safely and also handle empty dataframe
    if len(df) > 0 and isinstance(df["entities"].dropna().iloc[0], str):
        print("Converting stringified lists to Python objects...")
        df["entities"] = df["entities"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip() else []
        )
    else:
        # Also ensure NaNs become []
        df["entities"] = df["entities"].apply(lambda x: x if isinstance(x, list) else [])

    # --- 2) Initialize the cleaner using your corpus (frequency map) ---
    print("Initializing Cleaner and building frequency map...")
    all_text = df["context"].fillna("").astype(str).tolist()
    cleaner = SmartHistoricalCleaner(all_documents_text=all_text)

    # --- 3) Safe wrapper for each row ---
    def clean_row_safe(row):
        text = row["context"]
        if pd.isna(text):
            text = ""
        else:
            text = str(text)

        entities = row["entities"]
        if not isinstance(entities, list):
            entities = []

        # Force entities to (surface_form, "ENT") pairs.
        formatted_entities = []
        for item in entities:
            # Your entities may look like: ("France","GPE") or ("France","GPE", 12) etc.
            if isinstance(item, (list, tuple)) and len(item) > 0 and isinstance(item[0], str):
                ent = item[0]
                if ent and ent.strip():
                    formatted_entities.append((ent, "ENT"))
            # If your entities are sometimes plain strings, support that too:
            elif isinstance(item, str) and item.strip():
                formatted_entities.append((item.strip(), "ENT"))

        return cleaner.clean_text(text, formatted_entities)

    # --- 4) Run with progress bar ---
    print("Starting cleaning process (this may take time)...")
    tqdm.pandas(desc="Cleaning Docs")

    df["cleaned_context"] = df.progress_apply(clean_row_safe, axis=1)

    # --- 5) Verify ---
    print("\n✅ Done! Preview of the new column:")
    print(df["cleaned_context"][1])

Converting stringified lists to Python objects...
Initializing Cleaner and building frequency map...
Loading SymSpell...
Loading BERT fill-mask...


Device set to use cpu


Starting cleaning process (this may take time)...


Cleaning Docs: 100%|██████████| 131921/131921 [02:19<00:00, 943.02it/s] 


✅ Done! Preview of the new column:
Surely he above the rest of his fellow mortals partakes of heaven here below of bliss which none but the virtuous ever claim Obituary B In France Gen de Rosemberg aged 83 formerly Marshal of France Grand Officer of the Legion of Honor and commander of the French troops in the United States during the Revolutionary war In Washington Hon Uriah Tracy Esq Senator of the United States from the State of Connecticut aged 54 his pall was supported by the heads of department and officers of government he had been sick at Washington since March last In Baltimore during the week ending the 18th ult 15 adults and 23 children In Philadelphia during the week ending 18th ult 24 adults and 40 children In New York during the week ending 18th ult 2 men 6 women 10 boys and 7 girls At Newark N J the Rev Dr Alexander Macauley aged 73 At Danvers Dr Amos Putnam aged 83 At Andover Mrs Susanna Symmes relict of the late Rev Dr Wm Symmes aged 79 At Salem Mass Widow Margaret Sw

### INDICES CREATION

We create our indices

In [ ]:
#please, use our pre-computed indices
if not UPLOAD_VARIABLES:
    INDEX_BASELINE_PATH = os.path.join(BASE_DIR, 'final_final_notebook_bm25_index_aftererror_trial')
    INDEX_FUZZY_PATH = os.path.join(BASE_DIR, 'final_final_notebook_fuzzy_index_aftererror_trial')
    INDEX_CORRECTIONS_PATH = os.path.join(BASE_DIR, 'final_corrections_index_1')
    index_baseline= to_index(INDEX_BASELINE_PATH,"docno","context")
    index_fuzzy= to_index(INDEX_FUZZY_PATH,"docno","fuzzy_context",stemming=True) #something is probably wrong, pay attention
    
    index_correction = to_index(INDEX_CORRECTIONS_PATH,"docno","cleaned")

    INDEX_ENTITY_PATH = os.path.join(BASE_DIR, 'final_entity_index_multi')
    text_fields=['cleaned','entity_names']
    temp=df
    docs= temp[['docno','cleaned','entity_names']].to_dict('records')
    metas= {'docno' : 50}
    indexer = pt.index.IterDictIndexer(INDEX_ENTITY_PATH,meta=metas, text_attrs=text_fields, fields=True)
            
    indexref = indexer.index(docs)
    index_multi = pt.IndexFactory.of(indexref) #our index

## COMPONENTS CREATION

In this notebook we take a modular approach in order to avoid repetitions and to better structure our analysys. If a component is to be used multiple times in different pipelines, it will be executed once and its results will get saved as a transformer, this is particularly useful when different pipelines have the same first stage retrieval component or contain fusion of different component. It also help structuring an ablation analysys.

In [318]:
queries_sliced= queries_df[:10000]
metrics = [MAP,'P_1',R@5,R@10,'ndcg_cut_5','ndcg_cut_10','P_5','P_10']

### Fuzzy Component
This component is based on Hiemestra on the fuzzified context field. The chose n-grams are characters 5-grams. the query gets processed accordingly

In [319]:
fuzzy_component = pt.terrier.Retriever(index_fuzzy, wmodel="Hiemstra_LM")
fuzzy = pt.apply.query(format_5g) >> fuzzy_component
fuzzy_results = fuzzy.transform(queries_sliced)
fuzzy_ = pt.Transformer.from_df(fuzzy_results,uniform=True) # save results as a transformer, so we can use them for base of second stage retrieval or fusion

### BM25 Component
BM25 over the context field

In [320]:
bm25= pt.BatchRetrieve(index_baseline,wmodel='BM25')
bm25_results=bm25.transform(queries_sliced)
bm25_ = pt.Transformer.from_df(bm25_results,uniform=True)

C:\Users\samue\AppData\Local\Temp\ipykernel_3948\591657224.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25= pt.BatchRetrieve(index_baseline,wmodel='BM25')


### Hiemestra Component

Hiemestra is a language model useful for IR. It calculates the conditional probability of having a query given a document.

In [321]:
himestra= pt.terrier.Retriever(index_baseline,wmodel='Hiemstra_LM',verbose=True)
himestra_results= himestra.transform(queries_sliced)
himestra_ = pt.Transformer.from_df(himestra_results,uniform=True)

TerrierRetr(Hiemstra_LM): 100%|██████████| 10000/10000 [02:23<00:00, 69.55q/s]


### BO1 Query Expansion

This component apply a bo1 query expansion on the context field. it takes Hiemstra as first stage retrieval result and then run hiemstra again

In [322]:
bo1_expander = pt.rewrite.Bo1QueryExpansion(index_baseline, fb_docs=2, fb_terms=10,verbose=True)
bo1_expand = (himestra_>> bo1_expander >> himestra)
bo1_results= bo1_expand.transform(queries_sliced)
bo1 = pt.Transformer.from_df(bo1_results,uniform=True)

TerrierRetr(Hiemstra_LM): 100%|██████████| 10000/10000 [03:06<00:00, 53.61q/s]


### BM25 Field
For our proposed idea, we use a fielded index, hence we'll use BM25F.
In order to work with our query weighting module (which augment the frequency of terms in the query instahead of appending weights) we need to tune the query terms saturation parameter in our BM25

In [323]:
bm25_fields = pt.terrier.Retriever(index_multi, wmodel="BM25F", controls={"bm25.k_3": 1000},verbose=True)
first_component= (pt.apply.query(extract_kw) >> pt.apply.query(transform_w) >> bm25_fields) 
f_results = first_component.transform(queries_sliced)
first_component_ = pt.Transformer.from_df(f_results,uniform=True)  # first component of the proposed idea pipeline

TerrierRetr(BM25F): 100%|██████████| 10000/10000 [1:08:00<00:00,  2.45q/s]


### Hiemestra Cleaned

Hiemestra on the context cleaned

In [324]:
hiemestra_c = pt.BatchRetrieve(index_correction,wmodel='Hiemstra_LM') #we never use this component for base retrieval, so we do not calculate it now

C:\Users\samue\AppData\Local\Temp\ipykernel_3948\3170904630.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  hiemestra_c = pt.BatchRetrieve(index_correction,wmodel='Hiemstra_LM') #we never use this component for base retrieval, so we do not calculate it now


Bo1 query expansion on the first component, then another retriever (in this case hiemstra cleaned)

In [325]:
bo1_expander_c = pt.rewrite.Bo1QueryExpansion(index_correction, fb_docs=2, fb_terms=10,verbose=True)
second_component= first_component_ >> bo1_expander

s_results= second_component.transform(queries_sliced)
second_component_temp = pt.Transformer.from_df(s_results,uniform=True) #query expansion is very expensive, we separate the process so we can use the expanded queries on other

hiem_on_sec = second_component_temp >> hiemestra_c

h_s_r= hiem_on_sec.transform(queries_sliced)
second_component_ = pt.Transformer.from_df(h_s_r,uniform=True)

Transformer: 100%|██████████| 10000/10000 [1:09:26<00:00,  2.40q/s]


## BASELINES

As baselines we selected:
- BM25 on context
- Hiemstra on context
- Query Expansion >> Hiemstra on context

In [326]:
baselines_experiment= pt.Experiment(
    [bm25_,himestra_,bo1],
    queries_sliced,
    qrels_df,
    eval_metrics= metrics,
    names=['bm25','hiemstra','bo1>hiemstra']
)


print(baselines_experiment)

           name        AP     P_1      P_5     P_10     R@5    R@10  \
0          bm25  0.708274  0.6344  0.15916  0.08404  0.7958  0.8404   
1      hiemstra  0.712646  0.6409  0.15966  0.08421  0.7983  0.8421   
2  bo1>hiemstra  0.692506  0.6116  0.15614  0.08253  0.7807  0.8253   

   ndcg_cut_5  ndcg_cut_10  
0    0.722170     0.736715  
1    0.726473     0.740713  
2    0.706691     0.721214  


## ADVANCED EXPERIMENTS

We conducted 3 advanced experiments and analysys of their components

### E1 Query Expansion + Anchor

We noticed that applying query expansion (both bo1 and rm3) bring downgrade in performances.
In order to mitigate the phenomenon of query drift and gain value from our expansion, we fuse our results with a solid retriever, Hiemstra in this case.
We can notice how the drop in performances reduces. This method will revel itself more effective in the next experiment, were more difersified signals gets employed.



In [342]:
QE_anchor_exp = pt.Experiment(
    [bo1,himestra_,bo1 + himestra_, bo1 + fuzzy_ + himestra_],
    queries_sliced,
    qrels_df,
    eval_metrics= metrics,
    names=['default query expansion','default hiemstra','query expansion + hiemstra','qe + fuzzy + hiemstra'],
    baseline=3 #we compare this new pipeline with the 2 relevant components to show that th final result is better
)

print(QE_anchor_exp)

                         name        AP     P_1      P_5     P_10     R@5  \
0     default query expansion  0.692506  0.6116  0.15614  0.08253  0.7807   
1            default hiemstra  0.712646  0.6409  0.15966  0.08421  0.7983   
2  query expansion + hiemstra  0.703926  0.6279  0.15826  0.08362  0.7913   
3       qe + fuzzy + hiemstra  0.793258  0.7339  0.17264  0.08967  0.8632   

     R@10  ndcg_cut_5  ndcg_cut_10   AP +  ...    R@5 p-value  R@10 +  R@10 -  \
0  0.8253    0.706691     0.721214  640.0  ...  6.271794e-150    92.0   806.0   
1  0.8421    0.726473     0.740713  688.0  ...  7.932881e-107   110.0   656.0   
2  0.8362    0.717999     0.732564  664.0  ...  1.497947e-123   102.0   707.0   
3  0.8967    0.805228     0.816149    NaN  ...            NaN     NaN     NaN   

    R@10 p-value  ndcg_cut_5 +  ndcg_cut_5 -  ndcg_cut_5 p-value  \
0  4.212415e-129         359.0        2112.0       4.207099e-279   
1   2.602728e-88         402.0        1835.0       2.266247e-204   
2  1

### E2 Proposed Idea
This is the experiment that we porposed in our initial submission.
It makes use of query weighting, query expansion and uses different signal. 
One major change that we applied in the implementation is not applying second stage retrieval.
The high computational cost of cross-encoders is hard to substain, we have showed that neural second stage retrieval brings improvements on basic retrieval experiment, but that is not true for more complex ones. We attempted to apply second stage retrieval of our fuzzy components (using 1000 queries to sustain computational costs), and we found out that insthead of improving performances, those were worsening, so we decided to not apply it to this complex pipeline, since we belive it outperforms a simple neural ranker. An idea for future development would be fuse reranking results (on a simple but effective baseline) in order to make use of that diversified signal.

In [ ]:


exp= pt.Experiment(
    [fuzzy_, first_component_, second_component_, second_component_ + fuzzy_ ,second_component_ +himestra_ ,fuzzy_ + himestra_ ,second_component_ + fuzzy_ + himestra_],
    queries_sliced,
    qrels_df,
    eval_metrics=metrics,
    names=['5-grams','fc: keywords >> bm25f', 'sc: fc >> bo1 >> hiemstra clean','sc + 5-grams','sc + himestra','5-grams + himestra','final: sc + 5-grams + hiemstra'],
    baseline=6
    

)
print(exp)

                              name        AP     P_1      P_5     P_10  \
0                          5-grams  0.776693  0.7154  0.16988  0.08854   
1            fc: keywords >> bm25f  0.608349  0.5174  0.14296  0.07768   
2  sc: fc >> bo1 >> hiemstra clean  0.589264  0.5007  0.13778  0.07419   
3                     sc + 5-grams  0.788334  0.7276  0.17184  0.08940   
4                    sc + himestra  0.690375  0.6131  0.15638  0.08314   
5               5-grams + himestra  0.790696  0.7312  0.17214  0.08940   
6   final: sc + 5-grams + hiemstra  0.793605  0.7343  0.17320  0.08967   

      R@5    R@10  ndcg_cut_5  ndcg_cut_10   AP +  ...    R@5 p-value  R@10 +  \
0  0.8494  0.8854    0.789061     0.800711  760.0  ...   1.416982e-20    74.0   
1  0.7148  0.7768    0.624218     0.644378  672.0  ...  4.287481e-307   109.0   
2  0.6889  0.7419    0.604275     0.621562  509.0  ...   0.000000e+00    65.0   
3  0.8592  0.8940    0.800341     0.811664  645.0  ...   1.794483e-09    33.0   
4 

In [ ]:

exp= pt.Experiment(
    [first_component_ + fuzzy_ + himestra_],
    queries_sliced,
    qrels_df,
    eval_metrics=metrics,
    names=['best pipeline']

)
print(exp)

            name        AP     P_1      P_5    P_10     R@5   R@10  \
0  best pipeline  0.800445  0.7434  0.17354  0.0902  0.8677  0.902   

   ndcg_cut_5  ndcg_cut_10  AP +  ... R@5 p-value R@10 + R@10 - R@10 p-value  \
0    0.811824     0.822955  None  ...        None   None   None         None   

  ndcg_cut_5 + ndcg_cut_5 - ndcg_cut_5 p-value ndcg_cut_10 + ndcg_cut_10 -  \
0         None         None               None          None          None   

  ndcg_cut_10 p-value  
0                None  

[1 rows x 33 columns]
